This notebook contains the code to retrain gene-specific linear models with single cell RNAseq data, which is normalized as counts per million.

In [1]:
import pandas as pd
from scipy import optimize
from scipy.optimize import curve_fit
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from sklearn.linear_model import TheilSenRegressor
from sklearn.datasets import make_regression
from sklearn.metrics import r2_score
import pickle

In [2]:
# single cell data
df_CPM = pd.read_csv('data/expression/immune_dict_dataset.tsv.gz', sep='\t', compression='gzip')
df_CPM = df_CPM.loc[df_CPM["Cell"].isin([cell for cell in list(dict.fromkeys(df_CPM["Cell"])) if "Mean" in cell])]

In [3]:
# TPM and protein copy data 
df = pd.read_csv('data/expression/whole_dataset.csv')
# Get data where we got TPM and protein count
df_data = df.loc[np.logical_and(np.invert(np.isnan(df["RNA dataset num."])),np.invert(np.isnan(df["Prot. count dataset num."])))]

In [4]:
# Cells for the linear models
cells_data = ['T4','T8','Tregs','NK']
cells_CPM = [cell + '__PBS__Mean' for cell in ['T_cell_CD4', 'T_cell_CD8', 'Treg', 'NK_cell']]
df_CPM_cells = df_CPM.loc[df_CPM["Cell"].isin(cells_CPM)]
df_data_cells = df_data.loc[df_data["Cell"].isin(cells_data)]

In [27]:
# Set linear model dataset
gene_list = ["IL2RA_HUMAN","IL2RB_HUMAN","IL2RG_HUMAN","STA5A_HUMAN","STA5B_HUMAN",]
df_lin = pd.DataFrame(columns=["Gene","Intercept","Slope","r2"])

In [28]:
# Generate linear models
for gene in gene_list:
    df_data_cells_gene = df_data_cells.loc[df_data_cells["Gene"] == gene].sort_values(by='Cell', key=lambda x: x.map({val: i for i, val in enumerate(cells_data)})).reset_index(drop=True)
    df_CPM_cells_gene = df_CPM_cells.loc[df_CPM_cells["Gene"] == gene].sort_values(by='Cell', key=lambda x: x.map({val: i for i, val in enumerate(cells_CPM)})).reset_index(drop=True)

    # Elimnate cells with no data on 1 of the dataframes
    not_found_data = list(set(cells_data).symmetric_difference(set(df_data_cells_gene["Cell"].values)))
    not_found_CPM = list(set(cells_CPM).symmetric_difference(set(df_CPM_cells_gene["Cell"].values)))
    for i in range(0,len(cells_data)):
        for not_found in not_found_data:
            if not_found == cells_data[i]:
                df_CPM_cells_gene = df_CPM_cells_gene.drop(i)
    for i in range(0,len(cells_CPM)):
        for not_found in not_found_CPM:
            if not_found == cells_CPM[i]:
                df_data_cells_gene = df_data_cells_gene.drop(i)
                
    TPM = df_CPM_cells_gene["Log2 TPM"].values
    Prot = df_data_cells_gene["Log10 Prot. count"].values
    if gene == "IL2RG_HUMAN":
        Prot_RG = Prot
    model = TheilSenRegressor(random_state=0).fit(np.array(TPM).reshape(-1, 1), np.array(Prot).reshape(-1, 1))
    df_lin.loc[len(df_lin.index)] = [gene, model.intercept_, model.coef_[0], r2_score(Prot, model.predict(np.array(TPM).reshape(-1, 1)))]
# Correct IL2RG_HUMAN
df_lin.loc[len(df_lin.index)-3] = ["IL2RG_HUMAN", np.mean(Prot_RG), 0, np.nan]
df_lin.to_csv('data/lin_models/linear_models_IL2_CPM.csv')

In [29]:
color_sim = [
    (68/255, 138/255, 255/255),
    (0, 150/255, 136/255),
    (139/255, 195/255, 74/255),
    (255/255, 193/255, 7/255),
    (255/255, 152/255, 0),
    (244/255, 67/255, 54/255),
    (173/255, 20/255, 87/255)
]

fig, ax = plt.subplots(1, 1, figsize=(9, 8), dpi=600)
i = 0
for gene in gene_list:
    df_data_cells_gene = df_data_cells.loc[df_data_cells["Gene"] == gene].sort_values(by='Cell', key=lambda x: x.map({val: i for i, val in enumerate(cells_data)})).reset_index(drop=True)
    df_CPM_cells_gene = df_CPM_cells.loc[df_CPM_cells["Gene"] == gene].sort_values(by='Cell', key=lambda x: x.map({val: i for i, val in enumerate(cells_CPM)})).reset_index(drop=True)

    # Elimnate cells with no data on 1 of the dataframes
    not_found_data = list(set(cells_data).symmetric_difference(set(df_data_cells_gene["Cell"].values)))
    not_found_CPM = list(set(cells_CPM).symmetric_difference(set(df_CPM_cells_gene["Cell"].values)))
    for j in range(0,len(cells_data)):
        for not_found in not_found_data:
            if not_found == cells_data[j]:
                df_CPM_cells_gene = df_CPM_cells_gene.drop(j)
    for j in range(0,len(cells_CPM)):
        for not_found in not_found_CPM:
            if not_found == cells_CPM[j]:
                df_data_cells_gene = df_data_cells_gene.drop(j)

    # Training x/y values
    TPM_train = df_CPM_cells_gene["Log2 TPM"].values
    Prot_train = df_data_cells_gene["Log10 Prot. count"].values

    # Fitted model
    intercept = df_lin.loc[df_lin["Gene"] == gene, "Intercept"].values[0]
    slope = df_lin.loc[df_lin["Gene"] == gene, "Slope"].values[0]

    TPM_line = np.linspace(0, 12, 40)
    Prot_line = intercept + TPM_line * slope

    # Label
    label = gene.split("_HUMAN")[0]

    # Plot regression line and training points
    ax.plot(
        TPM_line,
        Prot_line,
        color=color_sim[i],
        linewidth=4,
        label=label
    )

    ax.scatter(
        TPM_train,
        Prot_train,
        s=60,
        color=color_sim[i],
        alpha=1
    )
    i += 1
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis="x", labelsize=22)
ax.tick_params(axis="y", labelsize=22)
plt.legend(loc="best", fontsize=18)
plt.ylabel("log10(Protein count)", fontsize=25)
plt.xlabel("log2(CPM)", fontsize=25)
plt.savefig(
    "figures/lin_models/corr_IL2_CPM.pdf",
    bbox_inches="tight",
    transparent=True
)
plt.close()